In [1]:
!pip install autogluon.tabular scikit-learn==1.5.2 "ray>=2.10.0,<2.45.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.3/487.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 22.8 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 7.1.0
    Uninstalling psutil-7.1.0:
      Successfully uninstalled psutil-7.1.0
  Attempting uninstall: ray
    Found existing installation: ray 2.49.2
    Uninstalling ray-2.49.2:
      Successfully uninstalled ray-2.49.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uni

In [2]:
!pip install autogluon.tabular[catboost]==1.4.0

In [3]:
import pandas as pd
import numpy as np
from scipy import stats

def road_risk(df):
    curvature = df['curvature'].fillna(0)
    lighting = df['lighting'].fillna("")
    weather = df['weather'].fillna("")
    speed_limit = df['speed_limit'].fillna(0)
    num_accidents = df['num_reported_accidents'].fillna(0)

    risk = (
        0.3 * curvature +
        0.2 * (lighting == "night").astype(int) +
        0.1 * (weather != "clear").astype(int) +
        0.2 * (speed_limit >= 60).astype(int) +
        0.1 * (num_accidents > 2).astype(int)
    )
    risk_min = risk.min()
    risk_max = risk.max()
    if risk_max == risk_min:
        return np.zeros_like(risk)
    else:
        return (risk - risk_min) / (risk_max - risk_min)

In [4]:
train_df = pd.read_csv("/kaggle/input/competition1/train.csv")
test_df = pd.read_csv("/kaggle/input/competition1/test.csv")

train_df['source'] = 'train'
test_df['source'] = 'test'

combined_df = pd.concat([train_df, test_df], ignore_index=True)
print(f"Original combined_df shape: {combined_df.shape}")

print("Creating base engineered features...")
combined_df['rule_based_risk'] = road_risk(combined_df)
combined_df['road_weather'] = combined_df['road_type'].astype(str) + '_' + combined_df['weather'].astype(str)
combined_df['road_light'] = combined_df['road_type'].astype(str) + '_' + combined_df['lighting'].astype(str)
combined_df['weather_light'] = combined_df['weather'].astype(str) + '_' + combined_df['lighting'].astype(str)
combined_df['group_id'] = combined_df['road_type'].astype(str) + '_' + combined_df['weather'].astype(str) + '_' + combined_df['lighting'].astype(str)
combined_df['speed_limit_x_curvature'] = combined_df['speed_limit'] * combined_df['curvature']
combined_df['speed_limit_div_lanes'] = combined_df['speed_limit'] / combined_df['num_lanes'].replace(0, 1)
combined_df['curvature_div_lanes'] = combined_df['curvature'] / combined_df['num_lanes'].replace(0, 1)

print("Creating deviation features selection...")
train_only_df = combined_df[combined_df['source'] == 'train'].copy()
road_type_avg_speed_map = train_only_df.groupby('road_type')['speed_limit'].mean()
road_type_avg_curve_map = train_only_df.groupby('road_type')['curvature'].mean()
combined_df['road_type_avg_speed'] = combined_df['road_type'].map(road_type_avg_speed_map)
combined_df['road_type_avg_curve'] = combined_df['road_type'].map(road_type_avg_curve_map)
global_avg_speed = train_only_df['speed_limit'].mean()
global_avg_curve = train_only_df['curvature'].mean()
combined_df['road_type_avg_speed'] = combined_df['road_type_avg_speed'].fillna(global_avg_speed)
combined_df['road_type_avg_curve'] = combined_df['road_type_avg_curve'].fillna(global_avg_curve)
combined_df['speed_deviation'] = combined_df['speed_limit'] - combined_df['road_type_avg_speed']
combined_df['curvature_deviation'] = combined_df['curvature'] - combined_df['road_type_avg_curve']
print("Base features, interactions, and deviation features created.")
print(f"Current combined_df shape: {combined_df.shape}")

Original combined_df shape: (690339, 15)
Creating base engineered features...
Creating deviation features selection...
Base features, interactions, and deviation features created.
Current combined_df shape: (690339, 27)


In [5]:
selected_features = [
    'rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather',
    'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light',
    'road_light', 'road_weather'
]

train_cols = selected_features + ['accident_risk', 'group_id']
print(f"Features for test set ({len(selected_features)}): {selected_features}")
print(f"Columns for train set ({len(train_cols)}): {train_cols}")

Features for test set (11): ['rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather', 'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light', 'road_light', 'road_weather']
Columns for train set (13): ['rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather', 'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light', 'road_light', 'road_weather', 'accident_risk', 'group_id']


In [6]:
print("Splitting combined_df back into train and test sets...")
ag_train_data = combined_df[combined_df['source'] == 'train'].copy()
ag_test_data = combined_df[combined_df['source'] == 'test'].copy()
print(f"ag_train_data shape: {ag_train_data.shape}")
print(f"ag_test_data shape: {ag_test_data.shape}")

print(f"\nSelecting {len(selected_features)} features...")
ag_train_data_features = ag_train_data[train_cols].copy()
ag_test_data_features = ag_test_data[selected_features].copy()

print("\nSelected features for training/testing.")
print(f"Final Training features shape: {ag_train_data_features.shape}")
print(f"Final Test features shape: {ag_test_data_features.shape}")
print("\nTraining Columns:", ag_train_data_features.columns.tolist())
print("Test Columns:", ag_test_data_features.columns.tolist())

Splitting combined_df back into train and test sets...
ag_train_data shape: (517754, 27)
ag_test_data shape: (172585, 27)

Selecting 11 features...

Selected features for training/testing.
Final Training features shape: (517754, 13)
Final Test features shape: (172585, 11)

Training Columns: ['rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather', 'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light', 'road_light', 'road_weather', 'accident_risk', 'group_id']
Test Columns: ['rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather', 'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light', 'road_light', 'road_weather']


In [7]:
import torch
from autogluon.tabular import TabularPredictor

label = 'accident_risk'
save_path = 'autogluon_model_v6'

predictor = TabularPredictor(
    label=label,
    path=save_path,
    eval_metric='root_mean_squared_error',
    problem_type='regression'
)

print("Starting AutoGluon Training...")
predictor.fit(
    ag_train_data_features,
    presets='best_quality',
    time_limit=3600*7,
    ag_args_fit={'groups': ag_train_data['group_id'], 'num_gpus': 2},
    num_bag_folds=10,
    dynamic_stacking=True
)
print("AutoGluon Training Finished")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.28 GB / 31.35 GB (93.4%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=10, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect stacked overfitting.
	Running DyStack for up to 6300s of the 25200s of remaining tim

Starting AutoGluon Training...


	Running DyStack sub-fit in a ray process to avoid memory leakage. Enabling ray logging (enable_ray_logging=True). Specify `ds_args={'enable_ray_logging': False}` if you experience logging issues.
2025-10-26 16:48:57,299	INFO worker.py:1852 -- Started a local Ray instance.
		Context path: "/kaggle/working/autogluon_model_v6/ds_sub_fit/sub_fit_ho"
(_dystack pid=188) Running DyStack sub-fit ...
(_dystack pid=188) Beginning AutoGluon training ... Time limit = 6284s
(_dystack pid=188) AutoGluon will save models to "/kaggle/working/autogluon_model_v6/ds_sub_fit/sub_fit_ho"
(_dystack pid=188) Train Data Rows:    460225
(_dystack pid=188) Train Data Columns: 12
(_dystack pid=188) Label Column:       accident_risk
(_dystack pid=188) Problem Type:       regression
(_dystack pid=188) Preprocessing data ...
(_dystack pid=188) Using Feature Generators to preprocess the data ...
(_dystack pid=188) Fitting AutoMLPipelineFeatureGenerator...
(_dystack pid=188) 	Available Memory:                    290

(_ray_fit pid=526) [1000]	valid_set's rmse: 0.0564957
(_ray_fit pid=526) [2000]	valid_set's rmse: 0.0564604


(_ray_fit pid=565) 	Training S1F2 with GPU, note that this may negatively impact model quality compared to CPU training.


...

(_ray_fit pid=4957) [1000]	valid_set's rmse: 0.0559356
(_ray_fit pid=4957) [2000]	valid_set's rmse: 0.0558736
(_ray_fit pid=4957) [3000]	valid_set's rmse: 0.0558217
(_ray_fit pid=4957) [4000]	valid_set's rmse: 0.0557712
(_ray_fit pid=4957) [5000]	valid_set's rmse: 0.0557136
(_ray_fit pid=4957) [6000]	valid_set's rmse: 0.0556691


(_ray_fit pid=4957) 	Ran out of time, early stopping on iteration 6194. Best iteration is:
(_ray_fit pid=4957) 	[6186]	valid_set's rmse: 0.0556581
(_ray_fit pid=4996) 	Training S1F10 with GPU, note that this may negatively impact model quality compared to CPU training.


(_ray_fit pid=4996) [1000]	valid_set's rmse: 0.0559039
(_ray_fit pid=4996) [2000]	valid_set's rmse: 0.0557946
(_ray_fit pid=4996) [3000]	valid_set's rmse: 0.0557376
(_ray_fit pid=4996) [4000]	valid_set's rmse: 0.0556856
(_ray_fit pid=4996) [5000]	valid_set's rmse: 0.0556512
(_ray_fit pid=4996) [6000]	valid_set's rmse: 0.0556151


(_ray_fit pid=4996) 	Ran out of time, early stopping on iteration 6333. Best iteration is:
(_ray_fit pid=4996) 	[6285]	valid_set's rmse: 0.0556068
(_dystack pid=188) 	-0.0558	 = Validation score   (-root_mean_squared_error)
(_dystack pid=188) 	736.22s	 = Training   runtime
(_dystack pid=188) 	167.48s	 = Validation runtime
(_dystack pid=188) Fitting model: RandomForestMSE_BAG_L2 ... Training model for up to 54.06s of the 52.72s of remaining time.
(_dystack pid=188) 	Warning: Model is expected to require 1151.2s to train, which exceeds the maximum time limit of 53.7s, skipping model...
(_dystack pid=188) 	Time limit exceeded... Skipping RandomForestMSE_BAG_L2.
(_dystack pid=188) Fitting model: CatBoost_BAG_L2 ... Training model for up to 37.64s of the 36.30s of remaining time.
(_dystack pid=188) 	Fitting 10 child models (S1F1 - S1F10) | Fitting with ParallelLocalFoldFittingStrategy (1.0 workers, per: cpus=1, gpus=2, memory=2.31%)
(_ray_fit pid=5156) 	Training S1F1 with GPU, note that thi

AutoGluon Training Finished


In [8]:
print(predictor.leaderboard())

                          model  score_val              eval_metric  \
0           WeightedEnsemble_L2  -0.056018  root_mean_squared_error   
1          LightGBM_r131_BAG_L1  -0.056049  root_mean_squared_error   
2          LightGBMLarge_BAG_L1  -0.056055  root_mean_squared_error   
3               LightGBM_BAG_L1  -0.056059  root_mean_squared_error   
4                XGBoost_BAG_L1  -0.056074  root_mean_squared_error   
5               CatBoost_BAG_L1  -0.056112  root_mean_squared_error   
6          CatBoost_r177_BAG_L1  -0.056120  root_mean_squared_error   
7             LightGBMXT_BAG_L1  -0.056274  root_mean_squared_error   
8           LightGBM_r96_BAG_L1  -0.056427  root_mean_squared_error   
9        NeuralNetFastAI_BAG_L1  -0.056543  root_mean_squared_error   
10    NeuralNetTorch_r79_BAG_L1  -0.056646  root_mean_squared_error   
11         ExtraTreesMSE_BAG_L1  -0.056868  root_mean_squared_error   
12       RandomForestMSE_BAG_L1  -0.057174  root_mean_squared_error   
13  Ne

In [9]:
print("Splitting combined_df back into train and test sets...")
ag_train_data = combined_df[combined_df['source'] == 'train'].copy()
ag_test_data = combined_df[combined_df['source'] == 'test'].copy()
print(f"ag_train_data shape: {ag_train_data.shape}")
print(f"ag_test_data shape: {ag_test_data.shape}")

print(f"\nSelecting {len(selected_features)} features...")
ag_train_data_features = ag_train_data[train_cols].copy()

test_cols = selected_features + ['group_id']
ag_test_data_features = ag_test_data[test_cols].copy()

print("\nSelected features for training/testing.")
print(f"Final Training features shape: {ag_train_data_features.shape}")
print(f"Final Test features shape: {ag_test_data_features.shape}")

print("\nTraining Columns:", ag_train_data_features.columns.tolist())
print("Test Columns:", ag_test_data_features.columns.tolist())

print(f"Using ag_test_data_features with shape: {ag_test_data_features.shape}")
print("\nGenerating predictions on the test features...")

predictions = predictor.predict(ag_test_data_features)
clipped_predictions = np.clip(predictions, 0, 1)

submission_df = pd.DataFrame({
    'id': test_df['id'].values,
    'accident_risk': clipped_predictions
})
submission_df.to_csv('submission_features.csv', index=False)
print("Saved 'submission_features.csv'.")

Splitting combined_df back into train and test sets...
ag_train_data shape: (517754, 27)
ag_test_data shape: (172585, 27)

Selecting 11 features...

Selected features for training/testing.
Final Training features shape: (517754, 13)
Final Test features shape: (172585, 12)

Training Columns: ['rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather', 'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light', 'road_light', 'road_weather', 'accident_risk', 'group_id']
Test Columns: ['rule_based_risk', 'curvature_deviation', 'num_reported_accidents', 'weather', 'lighting', 'speed_deviation', 'holiday', 'public_road', 'weather_light', 'road_light', 'road_weather', 'group_id']
Using ag_test_data_features with shape: (172585, 12)

Generating predictions on the test features...
Saved 'submission_features.csv'.


In [10]:
'''from sklearn.linear_model import LassoCV
from autogluon.tabular import TabularPredictor

predictor_path = '/kaggle/input/autogloun-full-comp1/autogluon_model_v5'
predictor = TabularPredictor.load(predictor_path)

l1_models = [
    'LightGBMLarge_BAG_L1',
    'XGBoost_BAG_L1',
    'LightGBM_r131_BAG_L1',
    'LightGBM_BAG_L1',
    'CatBoost_BAG_L1',
    'CatBoost_r177_BAG_L1',
    'CatBoost_r9_BAG_L1',
    'LightGBMXT_BAG_L1',
    'LightGBM_r96_BAG_L1',
    'ExtraTreesMSE_BAG_L1'
]

print(f"Identified {len(l1_models)} L1 models for blending.")

# Create Meta-Training Set
# We must get OOF preds for each model individually
print("Creating OOF meta-training features...")
meta_train_preds = {}
for model_name in l1_models:
    print(f"  Getting OOF for: {model_name}")
    meta_train_preds[model_name] = predictor.predict_oof(model=model_name)

X_meta_train = pd.DataFrame(meta_train_preds)
print("Meta-training features created.")

y_meta_train = train_df['accident_risk']

# Create Meta-Test Set
# We use ag_test_features (from cell 8) which has all FE applied
print("Creating meta-test features...")
meta_test_preds = {}
for model_name in l1_models:
    print(f"  Predicting with: {model_name}")
    meta_test_preds[model_name] = predictor.predict(ag_test_features, model=model_name)

X_meta_test = pd.DataFrame(meta_test_preds)
print("Meta-test features created.")'''

'from sklearn.linear_model import LassoCV\nfrom autogluon.tabular import TabularPredictor\n\npredictor_path = \'/kaggle/input/autogloun-full-comp1/autogluon_model_v5\'\npredictor = TabularPredictor.load(predictor_path)\n\nl1_models = [\n    \'LightGBMLarge_BAG_L1\',\n    \'XGBoost_BAG_L1\',\n    \'LightGBM_r131_BAG_L1\',\n    \'LightGBM_BAG_L1\',\n    \'CatBoost_BAG_L1\',\n    \'CatBoost_r177_BAG_L1\',\n    \'CatBoost_r9_BAG_L1\',\n    \'LightGBMXT_BAG_L1\',\n    \'LightGBM_r96_BAG_L1\',\n    \'ExtraTreesMSE_BAG_L1\'\n]\n\nprint(f"Identified {len(l1_models)} L1 models for blending.")\n\n# Create Meta-Training Set\n# We must get OOF preds for each model individually\nprint("Creating OOF meta-training features...")\nmeta_train_preds = {}\nfor model_name in l1_models:\n    print(f"  Getting OOF for: {model_name}")\n    meta_train_preds[model_name] = predictor.predict_oof(model=model_name)\n\nX_meta_train = pd.DataFrame(meta_train_preds)\nprint("Meta-training features created.")\n\ny_meta_

In [11]:
'''from sklearn.linear_model import RidgeCV

print("Training RidgeCV stacker...")
alphas_to_try = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 20.0]

ridge_stacker = RidgeCV(alphas=alphas_to_try, cv=10)
ridge_stacker.fit(X_meta_train, y_meta_train)

print("RidgeCV stacker training complete.")
print(f"Best alpha found: {ridge_stacker.alpha_}")

print("\n--- RidgeCV Coefficients (Weights) ---")
print(pd.Series(ridge_stacker.coef_, index=l1_models))'''

'from sklearn.linear_model import RidgeCV\n\nprint("Training RidgeCV stacker...")\nalphas_to_try = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 20.0]\n\nridge_stacker = RidgeCV(alphas=alphas_to_try, cv=10)\nridge_stacker.fit(X_meta_train, y_meta_train)\n\nprint("RidgeCV stacker training complete.")\nprint(f"Best alpha found: {ridge_stacker.alpha_}")\n\nprint("\n--- RidgeCV Coefficients (Weights) ---")\nprint(pd.Series(ridge_stacker.coef_, index=l1_models))'

In [12]:
'''print("Generating final predictions with RidgeCV stacker...")
final_ridge_predictions = ridge_stacker.predict(X_meta_test)
clipped_ridge_preds = np.clip(final_ridge_predictions, 0, 1)
ridge_submission_df = pd.DataFrame({
    'id': test_df['id'].values,
    'accident_risk': clipped_ridge_preds
})

ridge_submission_df.to_csv('ridge_stack_submission.csv', index=False)
print("Saved 'ridge_stack_submission.csv'.")'''

'print("Generating final predictions with RidgeCV stacker...")\nfinal_ridge_predictions = ridge_stacker.predict(X_meta_test)\nclipped_ridge_preds = np.clip(final_ridge_predictions, 0, 1)\nridge_submission_df = pd.DataFrame({\n    \'id\': test_df[\'id\'].values,\n    \'accident_risk\': clipped_ridge_preds\n})\n\nridge_submission_df.to_csv(\'ridge_stack_submission.csv\', index=False)\nprint("Saved \'ridge_stack_submission.csv\'.")'